In [0]:
# Databricks notebook source
# /// script
# [tool.databricks.environment]
# environment_version = "5"
# ///

# MAGIC %md
# MAGIC # Target Validation 실행
# MAGIC
# MAGIC 현재 개별 E2E 테스트를 위해 `COMPLAINT` Target만 검증한다.

# COMMAND ----------

SAVE = True  # False면 저장 없이 결과만 확인

# COMMAND ----------

import os
import sys

for _p in sys.path:
    _parent = os.path.dirname(_p)
    if os.path.isdir(os.path.join(_parent, "src")) and _parent not in sys.path:
        sys.path.insert(0, _parent)
        break

try:
    import src.gold.gold_validation_config as gcfg
    import src.gold.gold_validation_runner as gv
except ModuleNotFoundError:
    import gold_validation_config as gcfg
    import gold_validation_runner as gv

validator = gv.TargetValidator.from_tables(spark)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 0. Job Run ID 수신

# COMMAND ----------

dbutils.widgets.text("job_run_id", "")
job_run_id = dbutils.widgets.get("job_run_id") or None

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. 검증 대상
# MAGIC
# MAGIC 현재 개별 E2E 테스트를 위해 COMPLAINT만 검증한다.

# COMMAND ----------

TARGET_TABLE = "COMPLAINT"
targets = [TARGET_TABLE]

print(f"🎯 Validation 대상: {TARGET_TABLE}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Target별 검증

# COMMAND ----------

summaries = {}

for target_table in targets:
    print(
        f"\n{'=' * 60}\n"
        f"[{target_table}] Validation 시작\n"
        f"{'=' * 60}"
    )

    candidate_table = gcfg.gold_candidate_table(target_table)

    if not spark.catalog.tableExists(candidate_table):
        raise RuntimeError(
            f"{candidate_table} 없음 - Mapping Execution을 먼저 실행하세요."
        )

    passed, failed, summary = validator.run(
        target_table,
        job_run_id=job_run_id,
    )

    summaries[target_table] = summary

    match = (
        summary["input_count"]
        == summary["loaded_count"] + summary["quarantined_count"]
    )

    print(
        f"입력 {summary['input_count']} = "
        f"적재 {summary['loaded_count']} + "
        f"격리 {summary['quarantined_count']} "
        f"{'✅ 대사 일치' if match else '❌ 불일치'}"
    )

    if summary["violations_by_rule"]:
        for rule_id, cnt in sorted(
            summary["violations_by_rule"].items(),
            key=lambda kv: -kv[1],
        ):
            print(f"    위반 {rule_id}: {cnt}건")

    if summary["relationship_rules_skipped"]:
        print(
            f"⚠️ 검증되지 않은 FK "
            f"(참조 테이블이 아직 없음): "
            f"{summary['relationship_rules_skipped']}"
        )

    if SAVE:
        tables = validator.save(passed, failed, summary)

        for label, tbl in tables.items():
            print(f"✅ {label}: {tbl}")
    else:
        print("SAVE=False: 저장하지 않았습니다.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. 결과 요약

# COMMAND ----------

print("Validation 완료:", list(summaries.keys()))

for target_table, summary in summaries.items():
    skipped = len(summary["relationship_rules_skipped"])

    print(
        f"  {target_table:10s} "
        f"적재={summary['loaded_count']:>6}  "
        f"격리={summary['quarantined_count']:>6}  "
        f"미검증 FK={skipped}"
    )